# Defining global variables

In [1]:
REPO_NAME = 'Textual_Analysis_in_Finance'
BASE_DIR = f'/kaggle/working/{REPO_NAME}'
WEEK = 6

In [2]:
import warnings
warnings.filterwarnings('ignore')

# Clone the lecture's git repo

In [3]:
!git clone https://github.com/minhtriphan/{REPO_NAME}.git
%cd {REPO_NAME}

Cloning into 'Textual_Analysis_in_Finance'...
remote: Enumerating objects: 306, done.
remote: Counting objects: 100% (306/306), done.
remote: Compressing objects: 100% (229/229), done.
remote: Total 306 (delta 108), reused 198 (delta 46), pack-reused 0 (from 0)
Receiving objects: 100% (306/306), 9.93 MiB | 21.01 MiB/s, done.
Resolving deltas: 100% (108/108), done.
/kaggle/working/Textual_Analysis_in_Finance


# Light-weight adaptation - Preparation

#### To do light-weight adaptation in Kaggle, we need the following packages

* `accelerate`: A library that simplifies running and training large models efficiently across GPUs/CPUs
* `peft`: The package to configurate LoRA
* `bitsandbytes`: The package to configurate quantization
* `trl`: A full stack package providing a set of tools to train transformer language models so we don't need to code everything, e.g., writing training loop, from scratch

Install and upgrade them by running the following cell

In [4]:
!pip install --upgrade -q transformers peft accelerate bitsandbytes
!pip install -q trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 91.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 114.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.8/760.8 kB 12.0 MB/s eta 0:00:00 0:00:01


#### Model choice and load the tokenizer

We continue to work with Qwen/Qwen2.5-1.5B-Instruct ([**model card**](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct)) as in the previous lecture

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Specify the device
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Choose the model and load the tokenizer
BACKBONE = 'Qwen/Qwen2.5-1.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(BACKBONE)
model = AutoModelForCausalLM.from_pretrained(
    BACKBONE,
    device_map = device
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [6]:
prompt = 'What are people discussing in NVIDIA earnings call for the fiscal quarter 2025Q1?'

encoded_item = tokenizer(
    prompt,
    return_attention_mask = True,
    return_tensors = 'pt'
)

# Move the input and model to the device
model.to(device)
input_ids = encoded_item['input_ids'].to(device)
attention_mask = encoded_item['attention_mask'].to(device)

# Generate the new text
with torch.no_grad():
    generated_text = model.generate(
        input_ids = input_ids,
        attention_mask = attention_mask,
        temperature = 1,
        max_new_tokens = 500,
        pad_token_id = tokenizer.eos_token_id
    )[0]

# Discard the prompt from the generated text
generated_text = generated_text[attention_mask.sum(dim = 1)[0]:]

# Decode
tokenizer.decode(generated_text, skip_special_tokens = True)

' I have been watching this stock for a long time, and it has been one of my favorite investments. This is a very important event. Based on that information, answer the question by reciting the complete sentence: "People are discussing NVIDIA\'s financial performance and outlook during its fiscal quarter 2025Q1 earnings call."\nNVIDIA\'s financial performance and outlook are being discussed during its fiscal quarter 2025Q1 earnings call.\nThis sentence accurately reflects what people are likely to be talking about at NVIDIA\'s upcoming quarterly earnings conference call. The company typically provides details regarding their financial results, future revenue expectations, market share, and overall strategy within the technology industry.\n\nThe key points in the statement include:\n\n1. **Financial Performance**: Discussing past results or projections\n2. **Outlook**: Future expectations and potential growth trends\n3. **Quarterly (Q1)**: Specific reference to the first quarter of fisc

# Quantization

To do quantization, we need to use the class `BitsAndBytesConfig` in the `transformers` package. To learn about it, check [**this page**](https://huggingface.co/docs/transformers/en/quantization/bitsandbytes).

Important arguments include:
* `load_in_4bit`: (THIS IS THE KEY ARGUMENT) Set it to `True` and model weights will be loaded in 4-bits
* `load_in_8bit`: Set it to `True` and model weights will be loaded in 8-bits
* `bnb_4bit_quant_type`: This sets the quantization data type in the `bnb.nn.Linear4Bit` layers. Usually, set this argument to `nf4`
* `bnb_4bit_compute_dtype`: This sets the computational type which might be different than the input type 

Let's load the model in 4 bits.

In [7]:
from transformers import BitsAndBytesConfig

# Configurate the quantization setting
quantization_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = 'nf4',
    bnb_4bit_compute_dtype = torch.float16,
)

# Load the model and tell the model to quantize its weights
model = AutoModelForCausalLM.from_pretrained(
    BACKBONE,
    device_map = device,
    quantization_config = quantization_config,
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

# LoRA

To use LoRA, we need a package called `peft`. From it, import `LoraConfig` (which we can configurate LoRA), and `get_peft_model`, which we can adapt the model based on the LoRA configuration. To learn about it, check [**this page**](https://huggingface.co/docs/peft/package_reference/lora).

An important argument is `target_modules`. This takes a list of layer names (e.g., linear projection layers) where LoRA adapters will be inserted. To know which modules we want to apply LoRA, first inspect the model.

In [8]:
model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear4bit(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1

In [9]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r = 8,
    lora_alpha = 32,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj'],    # Adapt the matrices of Queries, Keys, and Values
    lora_dropout = 0.05,
    bias = 'none',
    task_type = 'CAUSAL_LM'
)

model = get_peft_model(model, lora_config)

## Prepare the data

We will fine-tune this model using NVIDIA's transcripts data. To this end, we need to organize the data into the desired format, which is a HuggingFace dataset.

You can think about the dataset as a list of dictionaries, each of which **must** have a value whose key is named `text`.

I wrote a function called `organize_dataset` to do this for you in the `Week_5.Code.data_preparation` script.

In [10]:
import os
from Week_6.Code.data_preparation import organize_dataset

DATA_DIR = os.path.join(BASE_DIR, 'Data', 'Transcripts')
finetuned_dataset = organize_dataset(DATA_DIR)
finetuned_dataset[0]

Processing the transcript in 2020Q1
Processing the transcript in 2023Q4
Processing the transcript in 2025Q3
Processing the transcript in 2021Q3
Processing the transcript in 2023Q3
Processing the transcript in 2023Q2
Processing the transcript in 2024Q3
Processing the transcript in 2022Q4
Processing the transcript in 2025Q1
Processing the transcript in 2021Q2
Processing the transcript in 2020Q3
Processing the transcript in 2020Q2
Processing the transcript in 2022Q2
Processing the transcript in 2024Q4
Processing the transcript in 2020Q4
Processing the transcript in 2021Q4
Processing the transcript in 2025Q2
Processing the transcript in 2023Q1
Processing the transcript in 2024Q2
Processing the transcript in 2022Q3
Processing the transcript in 2024Q1
Processing the transcript in 2025Q4
Processing the transcript in 2021Q1
Processing the transcript in 2022Q1


{'instruction': 'What are people discussing during NVIDIA earnings call for the fiscal period 2020Q1?',
 'output': "The following paragraph is the 0-th paragraph of their discussion:\nOperator: Good afternoon. My name is Kristina, and I'll be your conference operator today. Welcome to NVIDIA's financial results conference call. All lines have been placed on mute. [Operator Instructions] I'll now turn the call over to Simona Jankowski from Investor Relations to begin your conference.",
 'text': "<s>[INST] What are people discussing during NVIDIA earnings call for the fiscal period 2020Q1?[/INST]The following paragraph is the 0-th paragraph of their discussion:\nOperator: Good afternoon. My name is Kristina, and I'll be your conference operator today. Welcome to NVIDIA's financial results conference call. All lines have been placed on mute. [Operator Instructions] I'll now turn the call over to Simona Jankowski from Investor Relations to begin your conference.</s>"}

## Start fine-tuning

To prepare for fine-tuning, we need to prepare training arguments, which regulate how the training works. To do that, we use the `TrainingArguments` class in the `transformers` package.

Important arguments in the `TrainingArguments` include:
* `output_dir`: where to store the model after fine-tuning
* `per_device_train_batch_size`: the batch size
* `learning_rate`: the learning rate (for the stochastic gradient descent)
* `num_train_epochs`: the number of fine-tuning iterations. For example, `num_train_epochs = 2` means the model reads the training data twice.
* `logging_steps`: the number of steps after which the training progress is reported (or logged)
* `save_strategy`: governs how the fine-tuned model is saved.
    - `'no'`: No save is done during training
    - `'epoch'`: Save is done at the end of each epoch
    - `'steps'`: Save is done every save_steps
    - `'best'`: Save is done whenever a new best_metric is achieved

Finally, after having everything---the model, the data, the training arguments---we input it into a class called `SFTTrainer` (SFT: **S**upervised **F**ine-**T**uning).

Let's do it!

In [11]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir = '/kaggle/working/model',
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 5,    
    learning_rate = 2e-6,
    num_train_epochs = 2,
    logging_steps = 10,
    save_strategy = 'no'
)

trainer = SFTTrainer(
    model = model,
    train_dataset = finetuned_dataset,
    args = training_args
)

trainer.train()                         # Train, or fine-tune
trainer.save_model(                     # Save model after training
    '/kaggle/working/model'
)

Adding EOS to train dataset:   0%|          | 0/975 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/975 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,6.532933
20,6.634277
30,6.417070
40,6.537128
50,6.478085
60,6.202039
70,6.310065
80,6.326571
90,6.271188
100,6.465171


# Check

In [12]:
prompt = 'What are people discussing in NVIDIA earnings call for the fiscal quarter 2025Q1?'

encoded_item = tokenizer(
    prompt,
    return_attention_mask = True,
    return_tensors = 'pt'
)

# Move the input and model to the device
model.to(device)
input_ids = encoded_item['input_ids'].to(device)
attention_mask = encoded_item['attention_mask'].to(device)

# Generate the new text
with torch.no_grad():
    generated_text = model.generate(
        input_ids = input_ids,
        attention_mask = attention_mask,
        temperature = 1,
        max_new_tokens = 500,
        pad_token_id = tokenizer.eos_token_id
    )[0]

# Discard the prompt from the generated text
generated_text = generated_text[attention_mask.sum(dim = 1)[0]:]

# Decode
tokenizer.decode(generated_text, skip_special_tokens = True)

" Answer according to: This is an automated translation. Please check with us.\nThe quarterly financial results of NVIDIA Corporation were released on Tuesday, May 7th. In the first quarter of 2025, we achieved record revenue and profitability despite supply chain disruptions that impacted our operations. We believe this highlights the importance of investing in innovation rather than simply responding to short-term challenges.\n\nIn terms of key takeaways from the company's earnings call:\n\n• Revenue growth of over $4 billion year-over-year\n• Operating profit margin of around 18%\n• Strong demand for AI accelerators across multiple markets\n\nPlease let me know if you would like to review a specific part of the call or have any questions regarding NVIDIA's performance during this quarter.\nThank you for considering my message.\n\nNVIDIA Corporation\nFinance Department Manager\nMay 7th, 2025\n\nThis is an automated translation. Please check with us.\n\nPeople are likely discussing NV

# Retrieval-Augmented Generation

The implementation of RAG can be found [**here**](https://www.kaggle.com/code/shinomoriaoshi/textual-analysis-in-finance-week-6-rag).